<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2005%20-%20Eigenvectors%3A%20What%20a%20Transformation%20Leaves%20Alone/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 05 — Eigenvectors: What a Transformation Leaves Alone · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

Chapter 4 ended on an observation. The shear $S=\begin{bmatrix}1&1\\0&1\end{bmatrix}$ turns
almost every arrow it touches — but not $\begin{bmatrix}1\\0\end{bmatrix}$, which comes out
exactly as it went in.

Which directions does a matrix leave pointing where they started, and what can we do once
we know?

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. $A = \begin{bmatrix}2&1\\1&2\end{bmatrix}$. Guess one direction it does **not** turn.
   (Hint: the matrix treats $x$ and $y$ symmetrically.)
2. How many unturned directions does a $90°$ rotation have?
3. Apply $A$ to a random vector 20 times, normalizing each time. Does it wander, settle
   on one direction, or blow up?
4. Run PCA on rooms (2–4) and area (800–1600) unscaled. Which feature will "explain"
   almost all the variance — and is that a fact about houses?

In [ ]:
# Step 3 — Intuition: most directions turn, a few do not.
import numpy as np
np.random.seed(0)

A = np.array([[2., 1.], [1., 2.]])

def angle_change(M, v):
    out = M @ v
    cos = (v @ out) / (np.linalg.norm(v) * np.linalg.norm(out))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

for v in [np.array([1., 0.]), np.array([1., .5]), np.array([1., 1.]), np.array([1., -1.])]:
    print(f"v = {str(v):>12}  turned by {angle_change(A, v):6.2f} degrees")

# Two of them did not turn at all. Those are the eigenvectors.

## Step 4 — The Mathematics Under Test

$$A\mathbf{v} = \lambda\mathbf{v}
\qquad
\det(A - \lambda I) = 0
\qquad
A = PDP^{-1}
\qquad
A = U\Sigma V^{T}$$

Step 5 checks every number the lecture claims.

In [ ]:
# Step 5 — Manual calculation: solve the characteristic equation by hand, then check.
# det(A - lambda I) = (2-l)^2 - 1 = l^2 - 4l + 3 = (l-3)(l-1)  ->  l = 3, 1
roots = np.roots([1, -4, 3])
print("characteristic roots:", np.sort(roots))
assert np.allclose(np.sort(roots), [1., 3.])

v1 = np.array([1., 1.])    # for lambda = 3
v2 = np.array([1., -1.])   # for lambda = 1
print("\nA @ [1, 1]  =", A @ v1, " = 3 * [1, 1]")
print("A @ [1,-1]  =", A @ v2, " = 1 * [1,-1]")
assert np.allclose(A @ v1, 3 * v1)
assert np.allclose(A @ v2, 1 * v2)

# NumPy agrees (it normalizes eigenvectors to length 1).
vals, vecs = np.linalg.eig(A)
print("\nnumpy eigenvalues:", np.sort(vals))
assert np.allclose(np.sort(vals), [1., 3.])

In [ ]:
# Step 5b — when there is no answer, the mathematics says so honestly (blog section 6).
shear  = np.array([[1., 1.], [0., 1.]])
rotate = np.array([[0., -1.], [1., 0.]])

sv, svec = np.linalg.eig(shear)
print("shear eigenvalues:", sv, " -> 1 twice")
print("shear eigenvectors (columns):\n", np.round(svec, 6))
print("both columns are the same direction: the shear is DEFECTIVE")
assert np.allclose(sv, [1., 1.])
assert abs(abs(svec[:, 0] @ svec[:, 1]) - 1) < 1e-6   # the two columns are parallel

rv, _ = np.linalg.eig(rotate)
print("\nrotation eigenvalues:", rv, " -> complex, no real direction survives")
assert np.all(np.abs(rv.imag) > 0.5)
# Correct, not a bug: a 90 degree rotation turns every real direction.

In [ ]:
# Step 5c — rebuild the matrix from its eigenvectors (blog sections 7-8).
P = np.array([[1., 1.], [1., -1.]])
D = np.diag([3., 1.])
P_inv = np.linalg.inv(P)
print("P D P^-1 =\n", P @ D @ P_inv)
assert np.allclose(P @ D @ P_inv, A)

# Powers become trivial: A^n = P D^n P^-1
A10_fast = P @ np.diag([3.**10, 1.**10]) @ P_inv
A10_slow = np.linalg.matrix_power(A, 10)
print("\nA^10 =\n", np.round(A10_fast).astype(int))
assert np.allclose(A10_fast, A10_slow)
assert np.array_equal(np.round(A10_fast).astype(int), [[29525, 29524], [29524, 29525]])

print(f"\n3^10 = {3**10},  1^10 = 1")
print("After ten applications the first direction is 59049x more important than the second.")

In [ ]:
# Step 6 — First implementation: find the dominant eigenvector without any eig() call.
# Repeatedly applying a matrix drives everything toward its dominant direction.

def power_iteration(M, steps=50):
    v = np.array([1., 0.])          # any starting direction that is not an eigenvector
    history = []
    for _ in range(steps):
        v = M @ v
        v = v / np.linalg.norm(v)   # renormalize so it cannot blow up
        history.append(v.copy())
    return v, history

v, history = power_iteration(A)
print("converged to:", np.round(v, 6))
print("expected    :", np.round(v1 / np.linalg.norm(v1), 6))
assert np.allclose(np.abs(v), np.abs(v1 / np.linalg.norm(v1)), atol=1e-5)

# The eigenvalue falls out too: how much does A stretch that direction?
print("\nimplied eigenvalue:", round(float(v @ (A @ v)), 6), " (expected 3.0)")
assert abs(v @ (A @ v) - 3.0) < 1e-5

In [ ]:
# Step 7 — Visualization: the two special directions, and everything else.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) a ring of arrows, before and after
angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
for t in angles:
    v0 = np.array([np.cos(t), np.sin(t)])
    out = A @ v0
    axes[0].arrow(0, 0, *v0, head_width=.05, color="lightsteelblue", length_includes_head=True)
    axes[0].arrow(0, 0, *out, head_width=.08, color="steelblue", alpha=.6, length_includes_head=True)
for vec, col, lab in [(v1, "crimson", "eigen 3"), (v2, "darkgreen", "eigen 1")]:
    u = vec / np.linalg.norm(vec)
    axes[0].arrow(0, 0, *(3 * u), head_width=.12, color=col, length_includes_head=True, label=lab)
    axes[0].arrow(0, 0, *(-3 * u), head_width=.12, color=col, length_includes_head=True)
axes[0].set_title("every direction turns, except two"); axes[0].legend(fontsize=8)
axes[0].set_aspect("equal"); axes[0].grid(alpha=.3)

# (b) power iteration converging
hist = np.array(history)
axes[1].plot(hist[:, 0], hist[:, 1], "o-", ms=3)
u = v1 / np.linalg.norm(v1)
axes[1].plot(*u, "*", ms=18, color="crimson")
axes[1].set_title("power iteration walks to the dominant eigenvector")
axes[1].set_aspect("equal"); axes[1].grid(alpha=.3)

# (c) how fast: the gap shrinks like (lambda2/lambda1)^n
err = [np.linalg.norm(np.abs(h) - np.abs(u)) for h in hist]
axes[2].semilogy(err)
axes[2].set_title("error, log scale — ratio 1/3 per step")
axes[2].set_xlabel("step"); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

In [ ]:
# Step 8 — The experiment: PCA on our four houses (blog section 10).
rooms = np.array([2., 2., 3., 4.])
area  = np.array([800., 1200., 900., 1600.])
X = np.column_stack([rooms, area])

def pca(M):
    centered = M - M.mean(axis=0)
    cov = (centered.T @ centered) / (len(M) - 1)
    vals, vecs = np.linalg.eigh(cov)          # eigh: covariance is symmetric
    order = np.argsort(vals)[::-1]
    return vals[order], vecs[:, order], cov

vals_raw, vecs_raw, cov_raw = pca(X)
print("RAW FEATURES")
print("  covariance:\n", np.round(cov_raw, 2))
print(f"  eigenvalues: {vals_raw}")
print(f"  PC1 explains {100 * vals_raw[0] / vals_raw.sum():.2f}% of the variance")
print(f"  PC1 direction: {np.round(vecs_raw[:, 0], 5)}  <- essentially the area axis")
assert 100 * vals_raw[0] / vals_raw.sum() > 99.99

In [ ]:
# Step 9 — Change exactly one thing: standardize the features first.
X_std = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
vals_std, vecs_std, cov_std = pca(X_std)

print("STANDARDIZED FEATURES")
print("  correlation matrix:\n", np.round(cov_std, 5))
print(f"  eigenvalues: {np.round(vals_std, 5)}")
print(f"  PC1 explains {100 * vals_std[0] / vals_std.sum():.2f}%")
print(f"  PC1 direction: {np.round(vecs_std[:, 0], 4)}  <- rooms and area rising together")

r = cov_std[0, 1]
assert abs(r - 0.70232) < 1e-4
assert np.allclose(np.sort(vals_std), np.sort([1 + r, 1 - r]))
assert abs(100 * vals_std[0] / vals_std.sum() - 85.12) < 0.05

# The singular values of X explain Chapter 2's condition number.
sigma = np.linalg.svd(X, compute_uv=False)
print(f"\nsingular values of X: {np.round(sigma, 4)}")
print(f"ratio  sigma1/sigma2      = {sigma[0]/sigma[1]:,.1f}")
print(f"square of that ratio      = {(sigma[0]/sigma[1])**2:,.0f}")
print(f"Chapter 2 measured a condition number of about 3,604,714.")
assert abs((sigma[0] / sigma[1])**2 - 3_604_714) / 3_604_714 < 0.01

## Step 10 — Observe

Against your Step 2 predictions:

1. $A$ leaves $(1,1)$ and $(1,-1)$ unturned, stretching them by **3** and **1**.
2. The rotation has **no** real eigenvalues — correct, since it turns everything.
3. Repeated application settles onto $(1,1)/\sqrt{2}$, the dominant eigenvector, with the
   error shrinking by a factor of $\lambda_2/\lambda_1 = 1/3$ every step.
4. Raw PCA gives area **100.00%** of the variance. That is a fact about **units**, not
   houses: area's numbers are ~400× bigger, and variance squares that.

After standardizing, PC1 explains **85.12%** and means something — rooms and area rising
together, which we can fairly call *overall size*.

## Step 11 — Explain

**Why powers are easy.** $A = PDP^{-1}$ means $A^n = PD^nP^{-1}$, because every inner
$P^{-1}P$ cancels. Powering a diagonal matrix just powers each entry, so $3^{10} = 59049$
while $1^{10} = 1$. After ten steps one direction dominates by a factor of 59,049 — which is
exactly why power iteration converges, and why the error falls like $(\lambda_2/\lambda_1)^n$.

**Why this is the exploding/vanishing gradient story.** A deep network applies matrices over
and over. If eigenvalues exceed 1 the signal explodes; if they fall below 1 it vanishes.
Chapters 39–40 meet this as a practical crisis; here it is just $\lambda^n$.

**Where Chapter 2's condition number came from.** The singular values of $X$ are
$\sigma_1 \approx 2334.5$ and $\sigma_2 \approx 1.23$. The curvature of the loss surface
involves $X^TX$, whose eigenvalues are $\sigma_i^2$ — so the condition number is
$(\sigma_1/\sigma_2)^2 \approx 3.6$ million, the exact number that made Chapter 2's data
untrainable.

> Chapter 2 measured the symptom. This chapter found the cause.

In [ ]:
# Step 12 — Challenges.

# LEVEL 2 (by hand first): eigenvalues of B = [[4,1],[2,3]].
# Check your answers against trace = 7 and det = 10 before running this.
B = np.array([[4., 1.], [2., 3.]])
print("trace:", np.trace(B), " det:", np.linalg.det(B))
print("eigenvalues:", np.linalg.eigvals(B))

# LEVEL 4 (Investigate): power iteration on a matrix whose eigenvalues are close.
# Compare how many steps A = [[2,1],[1,2]] needs against [[2,0],[0,1.999]].
# Explain the difference using the ratio lambda2/lambda1.

# YOUR CODE HERE


# LEVEL 5 (Design): compress a 1000x1000 image keeping only 50 numbers per row,
# using the SVD. What do you keep, what do you discard, how do you reconstruct,
# and what kind of image would your scheme ruin?

# YOUR CODE HERE

## Step 13 — Reflection

- [ ] I solved $\det(A - \lambda I) = 0$ by hand and got 3 and 1.
- [ ] I can say why a rotation has no real eigenvectors without calling it a bug.
- [ ] I can explain why $A^{10}$ is easy once you know $P$ and $D$.
- [ ] I can connect $\lambda^n$ to exploding and vanishing gradients.
- [ ] I know why PCA on unscaled data is a statement about units.
- [ ] I can explain where Chapter 2's 3.6 million came from.

### The question this chapter leaves open

Part I is finished. We can describe a number, a list, a table, a transformation, and the
skeleton a transformation is built around.

Every one of those describes **structure that sits still**. But Chapter 1 needed something
else and we faked it: when we asked *"which way should I nudge $w$?"*, we computed the loss
at two nearby points, took a difference, divided by the gap, and watched the gap shrink to
nothing. Nothing in Part I can express that.

➡️ **Next:** [Chapter 06 — Derivatives: The Compass for Learning](<../Lecture 06 - Derivatives: The Compass for Learning/blog.md>)